## Resume Hiring Predictor - PDF Version

In [ ]:
!pip install PyPDF2

In [ ]:

import os
import logging
import queue
import threading
import time
import gradio as gr
from PyPDF2 import PdfReader
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd
import plotly.graph_objects as go

In [ ]:
Upload resumes (PDFs) and predict likelihood of matching a specific job requirement.
 
# Helper Functions

class QueueHandler(logging.Handler):
    def __init__(self, log_queue):
        super().__init__()
        self.log_queue = log_queue

    def emit(self, record):
        self.log_queue.put(self.format(record))



Set up loggin

In [ ]:
def setup_logging(log_queue):
    handler = QueueHandler(log_queue)
    formatter = logging.Formatter("[%(asctime)s] %(message)s", datefmt="%H:%M:%S")
    handler.setFormatter(formatter)
    logger = logging.getLogger()
    logger.addHandler(handler)
    logger.setLevel(logging.INFO)


def html_for(log_data):
    return '<br>'.join(log_data[-20:])

In [ ]:
# Resume Parsing & Matching

def extract_text_from_pdf(file_path):
    reader = PdfReader(file_path)
    text = ""
    for page in reader.pages:
        text += page.extract_text() + "\n"
    return text

def score_resume(resume_text, job_requirements):
    """Compute match score using TF-IDF cosine similarity"""
    vectorizer = TfidfVectorizer(stop_words='english')
    vectors = vectorizer.fit_transform([resume_text, job_requirements])
    similarity = cosine_similarity(vectors[0:1], vectors[1:2])[0][0]
    return round(similarity * 100, 2)  # % match


In [ ]:

# Main App Class

class HiringApp:
    def __init__(self):
        self.candidates = []

    def load_initial(self):
        """Return empty table and logs"""
        df = pd.DataFrame(columns=["Candidate", "Match %"])
        return [], "", df

    def process_pdf(self, pdf_file, job_requirements, log_data):
        """Process uploaded PDF and compute match"""
        log_queue = queue.Queue()
        setup_logging(log_queue)

        def worker():
            logging.info(f"Processing PDF: {pdf_file.name}")
            try:
                text = extract_text_from_pdf(pdf_file.name)
                score = score_resume(text, job_requirements)
                candidate_name = os.path.basename(pdf_file.name).replace(".pdf", "")
                self.candidates.append({"Candidate": candidate_name, "Match %": score})
                logging.info(f"Candidate '{candidate_name}' match score: {score}%")
            except Exception as e:
                logging.error(f"Error processing PDF: {str(e)}")

        thread = threading.Thread(target=worker)
        thread.start()

        # Update logs and table while processing
        while thread.is_alive():
            try:
                message = log_queue.get_nowait()
                log_data.append(message)
            except queue.Empty:
                pass
            df = pd.DataFrame(self.candidates)
            yield log_data, html_for(log_data), df
            time.sleep(0.1)

        df = pd.DataFrame(self.candidates)
        yield log_data, html_for(log_data), df

    def run(self):
        with gr.Blocks(title="Resume Hiring Predictor") as ui:
            log_data = gr.State([])

            gr.Markdown("## Resume Hiring Predictor - Upload PDF to check match with job requirements")

            with gr.Row():
                pdf_upload = gr.File(file_types=[".pdf"])
                job_req_input = gr.Textbox(label="Job Requirements", placeholder="Enter job requirements here...")

            table_output = gr.Dataframe(
                headers=["Candidate", "Match %"], row_count=5, max_height=300
            )
            logs = gr.HTML()

            ui.load(self.load_initial, inputs=[], outputs=[log_data, logs, table_output])

            pdf_upload.upload(self.process_pdf, inputs=[pdf_upload, job_req_input, log_data], outputs=[log_data, logs, table_output])

        ui.launch(share=False, inbrowser=True)


In [ ]:
# Run the app
app = HiringApp()
app.run()